# Tutorial: Using the Insights Feature in Google Cloud Contact Center AI

This notebook provides a comprehensive guide to using the `insights.py` module, which offers a simplified interface for interacting with the Google Cloud Contact Center AI Insights (CCAI Insights) API.

CCAI Insights helps businesses analyze customer interactions (like phone calls and chats) to extract valuable insights, identify trends, and improve customer experience. This module wraps the core functionalities, allowing you to:

-   **Manage Global Settings:** Configure default analysis behaviors, data retention, Pub/Sub notifications, language, speech recognizers, and DLP (Data Loss Prevention) settings.
-   **Ingest Conversation Data:** Upload single audio/transcript files or bulk conversation data into CCAI Insights.
-   **Perform Analysis:** Trigger analysis on individual conversations or in bulk to extract sentiment, entities, intents, topics, and more.
-   **Export Data:** Export processed insights data to BigQuery for further analysis and reporting.
-   **Manage Authorized Views:** Create and manage secure views of your conversation data for different user groups or applications.
-   **Handle Long-Running Operations:** Monitor and manage asynchronous API operations.

Let's get started!

## Setup and Installation

In [ ]:
# Install necessary libraries
# google-cloud-contact-center-insights is the official Google Cloud client library.
# strenum is used for string enums in the insights.py module.
# requests is used by the mock conidk.core.base.Request class.
!pip install --upgrade google-cloud-contact-center-insights google-api-core strenum requests

print("Required libraries installed.")

### Mock `conidk.core.base` Module
The `insights.py` module relies on an internal Google library, `conidk.core.base`, for common functionalities like authentication, configuration, and generalized API requests. Since `conidk` is not publicly available, we will create mock classes and constants here to allow the `insights.py` module to be imported and demonstrated within this notebook without modification to its source code.

These mocks will simulate the expected behavior of `conidk.core.base.Auth`, `conidk.core.base.Config`, `conidk.core.base.Request`, `conidk.core.base.Methods`, and `conidk.core.base.SECONDS_IN_A_YEAR`.

In [ ]:
import os
import sys
from enum import Enum
import requests
from typing import Dict, Optional, List, MutableMapping
from random import randint

# --- Mock conidk.core.base classes and constants ---

class MockAuth:
    """Mock Auth class for authentication."""
    def __init__(self):
        pass

class MockConfig:
    """Mock Config class for API configuration."""
    def __init__(self):
        pass

    def set_insights_endpoint(self):
        """Mock method to set the Insights API endpoint.
        For insights_v1, returning None means the default endpoint will be used by the client.
        """
        return None

MOCK_SECONDS_IN_A_YEAR = 365 * 24 * 60 * 60  # Used for TTL calculation

class MockMethods(Enum):
    """Mock Enum for HTTP methods."""
    GET = "GET"
    POST = "POST"
    PUT = "PUT"
    DELETE = "DELETE"

class MockRequest:
    """Mock Request class for simulating API calls, specifically for AuthorizedViews."""
    def __init__(self, project_id: str, location: str, auth: MockAuth, config: MockConfig, base_url: str):
        self.project_id = project_id
        self.location = location
        self.auth = auth
        self.config = config
        self.base_url = base_url
        self._session = requests.Session()

    def make(self, endpoint: str, method: MockMethods, payload: Optional[Dict] = None) -> Optional[requests.Response]:
        """Simulates an API request and returns a mock response."""
        url = f"{self.base_url}{endpoint}"
        print(f"[Mock Request] {method.value} {url}\n  Payload: {payload}")

        class MockHttpResponse:
            def __init__(self, status_code, json_data):
                self.status_code = status_code
                self._json_data = json_data

            def json(self):
                return self._json_data

            @property
            def ok(self):
                return 200 <= self.status_code < 300

        if method == MockMethods.POST:
            # Simulate creation of a resource and generate a name
            resource_display_name = payload.get('displayName', f'mock-resource-{randint(1000, 9999)}')
            resource_id_slug = resource_display_name.lower().replace(' ', '-')
            
            if 'authorizedViewSets' in endpoint and endpoint.endswith('authorizedViewSets'):
                resource_path = f"projects/{self.project_id}/locations/{self.location}/authorizedViewSets/{resource_id_slug}"
            elif 'authorizedViews' in endpoint and endpoint.endswith('authorizedViews'):
                parent_set_id = url.split('/')[-2] # Extract from base_url/endpoint
                resource_path = f"projects/{self.project_id}/locations/{self.location}/authorizedViewSets/{parent_set_id}/authorizedViews/{resource_id_slug}"
            else:
                resource_path = f"{url}/{resource_id_slug}"
            
            return MockHttpResponse(200, {"name": resource_path, **payload})
        
        elif method == MockMethods.GET:
            if endpoint.endswith('/authorizedViewSets'):
                return MockHttpResponse(200, {"authorizedViewSets": [
                    {"name": f"projects/{self.project_id}/locations/{self.location}/authorizedViewSets/mock-view-set-1", "displayName": "Mock View Set 1"},
                    {"name": f"projects/{self.project_id}/locations/{self.location}/authorizedViewSets/mock-view-set-2", "displayName": "Mock View Set 2"}
                ]})
            elif endpoint.endswith('/authorizedViews'):
                parent_set_id = url.split('/')[-2] # Extract from base_url/endpoint
                return MockHttpResponse(200, {"authorizedViews": [
                    {"name": f"projects/{self.project_id}/locations/{self.location}/authorizedViewSets/{parent_set_id}/authorizedViews/mock-view-1", "displayName": "Mock View 1"},
                    {"name": f"projects/{self.project_id}/locations/{self.location}/authorizedViewSets/{parent_set_id}/authorizedViews/mock-view-2", "displayName": "Mock View 2"}
                ]})
            elif 'authorizedViewSets/' in endpoint and 'authorizedViews/' not in endpoint and len(endpoint.split('/')) == 2: # Get specific view set
                view_set_id = endpoint.split('/')[-1]
                return MockHttpResponse(200, {"name": f"projects/{self.project_id}/locations/{self.location}/authorizedViewSets/{view_set_id}", "displayName": f"Mock Single View Set {view_set_id}"})
            elif 'authorizedViewSets/' in endpoint and 'authorizedViews/' in endpoint and len(endpoint.split('/')) == 4: # Get specific view
                view_set_id = endpoint.split('/')[-3]
                view_id = endpoint.split('/')[-1]
                return MockHttpResponse(200, {"name": f"projects/{self.project_id}/locations/{self.location}/authorizedViewSets/{view_set_id}/authorizedViews/{view_id}", "displayName": f"Mock Single View {view_id}"})
            else:
                return MockHttpResponse(200, {"status": "success", "message": "Mocked GET response"})
        
        elif method == MockMethods.DELETE:
            return MockHttpResponse(200, {})  # Empty dict for successful delete
        
        else:
            return MockHttpResponse(200, {"status": "success", "message": "Mocked response"})

# Dynamically create the conidk.core.base module structure in sys.modules
# This allows the insights.py code to import 'from conidk.core import base' directly.

class _DummyModule(sys.modules[__name__].__class__):
    pass

sys.modules['conidk'] = _DummyModule('conidk')
sys.modules['conidk.core'] = _DummyModule('conidk.core')
sys.modules['conidk.core.base'] = _DummyModule('conidk.core.base')

# Populate the mock base module with our mock classes and constants
sys.modules['conidk.core.base'].Auth = MockAuth
sys.modules['conidk.core.base'].Config = MockConfig
sys.modules['conidk.core.base'].SECONDS_IN_A_YEAR = MOCK_SECONDS_IN_A_YEAR
sys.modules['conidk.core.base'].Methods = MockMethods
sys.modules['conidk.core.base'].Request = MockRequest

print("Mock `conidk.core.base` module created and populated.")

### Google Cloud Authentication
Ensure you are authenticated to Google Cloud. If running in Colab, this typically involves authenticating your Google account. If running locally, you might use `gcloud auth application-default login` in your terminal or set environment variables for service account credentials.

In [ ]:
try:
    from google.colab import auth as colab_auth
    colab_auth.authenticate_user()
    print("Authenticated to Google Cloud using Colab auth.")
except ImportError:
    print("Not in Colab environment. Assuming authentication is handled via gcloud or environment variables.")
    # For local execution, ensure your gcloud CLI is authenticated:
    # !gcloud auth application-default login
except Exception as e:
    print(f"Could not authenticate: {e}. Please ensure gcloud is configured or you are running in Colab.")

### Define Project Variables
Replace the placeholder values with your Google Cloud Project ID and desired Location. The location is typically a Google Cloud region where your CCAI Insights resources reside (e.g., `us-central1`).

In [ ]:
PROJECT_ID = "YOUR_PROJECT_ID" # @param {type:"string"}
LOCATION = "us-central1" # @param {type:"string"} E.g., "us-central1", "global"

# The parent resource name for most API calls, formatted as projects/{PROJECT_ID}/locations/{LOCATION}
PARENT = f"projects/{PROJECT_ID}/locations/{LOCATION}"

print(f"Project ID: {PROJECT_ID}")
print(f"Location: {LOCATION}")
print(f"Parent Resource: {PARENT}")

if PROJECT_ID == "YOUR_PROJECT_ID":
    print("\n*** IMPORTANT: Please update YOUR_PROJECT_ID with your actual Google Cloud Project ID. ***")

## The `insights.py` Module Code
Below is the complete source code for the `insights.py` module. We'll run this cell to define all the classes and enums so they can be used in subsequent examples. Note that the original `from conidk.core import base` import will now successfully resolve to our mock module defined above.

In [ ]:
# Copyright 2025 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

"""Wrapper for Contact Center AI Insights API interactions.
 
This module provides a simplified interface for interacting with the Google Cloud
Contact Center AI Insights API. It includes classes for managing settings,
ingesting conversation data, performing analysis, exporting data, and managing
authorized views.
"""

import enum
from random import randint
from typing import Dict, MutableMapping, Optional, List

from google.cloud import contact_center_insights_v1
from google.cloud.contact_center_insights_v1 import types
from google.longrunning.operations_pb2 import Operations, Operation, CancelOperationRequest #type: ignore  # pylint: disable=E0611
from google.protobuf import duration_pb2 #type: ignore
from google.protobuf.timestamp_pb2 import Timestamp  #type: ignore # pylint: disable=E0611
from strenum import StrEnum

from conidk.core import base # This import is now handled by our mock `sys.modules` definitions

class Masks(StrEnum):
    """Enum for supported field masks in update requests."""
    #Note that the types has to be camelcase becaus it's the api requirement
    ANALYSIS = "analysisConfig"
    TTL = "conversationTtl"
    PUBSUB = "pubsubNotificationSettings"
    LANGUAGE = "languageCode"
    SPEECH_RECOGNIZER = "speechConfig.speechRecognizer"
    DLP = "redactionConfig.inspectTemplate,redactionConfig.deidentifyTemplate"

class Annotators(StrEnum):
    """Enum for supported annotators."""
    QAI = "QAI"
    INSIGHTS = "INSIGHTS"
    TOPIC_MODEL = "TOPIC MODEL"
    SUMMARIZATION = "SUMMARIZATION"

class AgentType(enum.Enum):
    """Enum for supported agent types."""
    HUMAN_AGENT = types.ConversationParticipant.Role.HUMAN_AGENT
    AUTOMATED_AGENT = types.ConversationParticipant.Role.AUTOMATED_AGENT

class Mediums(enum.Enum):
    """Enum for supported conversation mediums."""
    PHONE_CALL = contact_center_insights_v1.Conversation.Medium.PHONE_CALL
    CHAT = contact_center_insights_v1.Conversation.Medium.CHAT

class Settings:
    """Manages Contact Center AI Insights settings."""

    def __init__(
        self,
        project_id: str,
        parent: str,
        auth: Optional[base.Auth] = None,
        config: Optional[base.Config] = None,
    ) -> None:
        """Initializes the Settings client.

        Args:
            project_id: The Google Cloud project ID.
            parent: The parent resource name (e.g., 'projects/p/locations/l').
            auth: An optional authentication object.
            config: An optional configuration object.
        """
        self.auth = auth or base.Auth()
        self.config = config or base.Config()

        self.parent = f"{parent}/settings"
        self.project_id = project_id
        self.client = contact_center_insights_v1.ContactCenterInsightsClient(
            client_options=self.config.set_insights_endpoint()
        )

    def _send_update_settings(
            self, request: types.UpdateSettingsRequest
    ) -> types.Settings:
        """Sends an update settings request.

        Args:
            request: The update settings request.
        
        Returns:
            The updated settings.
        """
        return self.client.update_settings(
            request=request
        )

    def _set_annotators(
        self,
        annotators: List[str],
    ) -> types.AnnotatorSelector:
        """Configures annotators for analysis.
 
        Args:
            annotators: A list of annotators to enable.

        Returns:
            An AnnotatorSelector object with the specified annotators enabled.
        """
        selected_annotators = types.AnnotatorSelector(
            run_interruption_annotator=False,
            run_silence_annotator=False,
            run_phrase_matcher_annotator=False,
            run_sentiment_annotator=False,
            run_entity_annotator=False,
            run_intent_annotator=False,
            run_issue_model_annotator=False,
            run_summarization_annotator=False,
        )

        for annotator in annotators:
            if annotator == Annotators.SUMMARIZATION:
                selected_annotators.run_summarization_annotator = True
            if annotator == Annotators.TOPIC_MODEL:
                selected_annotators.run_issue_model_annotator = True
            if annotator == Annotators.INSIGHTS:
                selected_annotators.run_intent_annotator = True
                selected_annotators.run_entity_annotator = True
                selected_annotators.run_sentiment_annotator = True
                selected_annotators.run_phrase_matcher_annotator = True
                selected_annotators.run_silence_annotator = True
                selected_annotators.run_interruption_annotator = True
            if annotator == Annotators.QAI:
                selected_annotators.run_qai_annotator = True

        return selected_annotators

    def update_global_auto_analysis(
        self,
        runtime_percentage: float,
        upload_percentage: float,
        analysis_annotators: List[str],
    ) -> types.Settings:
        """Updates the global auto-analysis settings.
 
        Args:
            runtime_percentage: Percentage of runtime conversations to analyze.
            upload_percentage: Percentage of uploaded conversations to analyze.
            analysis_annotators: A list of annotators to use for analysis.

        Returns:
            The updated settings.
        """
        request = types.UpdateSettingsRequest(
            settings=types.Settings(
                name=self.parent,
                analysis_config=types.Settings.AnalysisConfig(
                    runtime_integration_analysis_percentage=runtime_percentage,
                    upload_conversation_analysis_percentage=upload_percentage,
                    annotator_selector=self._set_annotators(analysis_annotators),
                ),
            ),
            update_mask=Masks.ANALYSIS,
        )
        return self._send_update_settings(request)

    def update_ttl(
        self,
        ttl_in_days: int,
    ) -> types.Settings:
        """Updates the conversation time-to-live (TTL).

        Args:
            ttl_in_days: The new TTL in days.

        Returns:
            The updated settings.
        """
        request = contact_center_insights_v1.UpdateSettingsRequest(
            settings=contact_center_insights_v1.Settings(
                name=self.parent,
                # pylint: disable-next=no-member
                conversation_ttl=duration_pb2.Duration(
                    seconds=ttl_in_days * base.SECONDS_IN_A_YEAR
                ),
            ),
            update_mask=Masks.TTL,
        )
        return self._send_update_settings(request)

    def update_pubsub(self, pub_sub_map: Dict[str, str]) -> types.Settings:
        """Updates the Pub/Sub notification settings.
 
        Args:
            pub_sub_map: A dictionary mapping notification types to Pub/Sub topics.

        Returns:
            The updated settings.
        """
        request = types.UpdateSettingsRequest(
            settings=types.Settings(
                name=self.parent, pubsub_notification_settings=pub_sub_map
            ),
            update_mask=Masks.PUBSUB,
        )
        return self._send_update_settings(request)

    def update_global_language(self, language_code: str) -> types.Settings:
        """Updates the default language for conversations.
        
        Args:
            language_code: The new default language code (e.g., 'en-US').

        Returns:
            The updated settings.
        """
        request = contact_center_insights_v1.UpdateSettingsRequest(
            settings=contact_center_insights_v1.Settings(
                name=self.parent, language_code=language_code
            ),
            update_mask=Masks.LANGUAGE,
        )
        return self._send_update_settings(request)

    def update_global_speech(
        self, speech_recognizer_path: str
    ) -> types.Settings:
        """Updates the default speech recognizer.
        
        Args:
            speech_recognizer_path: The resource name of the speech recognizer.

        Returns:
            The updated settings.
        """

        request = types.UpdateSettingsRequest(
            update_mask=Masks.SPEECH_RECOGNIZER,
            settings=types.Settings(
                name=self.parent,
                speech_config=types.SpeechConfig(
                    speech_recognizer=speech_recognizer_path
                ),
            ),
        )
        return self._send_update_settings(request)

    def update_global_dlp(
        self, inspect_template: str, deidentify_template: str
    ) -> types.Settings:
        """Updates the global DLP (Data Loss Prevention) settings.
       
       Args:
            inspect_template: The resource name of the DLP inspect template.
            deidentify_template: The resource name of the DLP de-identify template.

        Returns:
            The updated settings.
        """

        request = types.UpdateSettingsRequest(
            settings=types.Settings(
                name=self.parent,
                redaction_config=types.RedactionConfig(
                    inspect_template=inspect_template,
                    deidentify_template=deidentify_template,
                ),
            ),
            update_mask=Masks.DLP,
        )
        return self._send_update_settings(request)

    def get(self) -> types.Settings:
        """Retrieves the current settings.

        Returns:
            The current settings.
        """
        return self.client.get_settings(
            request=types.GetSettingsRequest(name=self.parent)
        )

class Ingestion:
    """Handles the ingestion of conversation data into Contact Center AI Insights."""

    def __init__(
        self,
        parent: str,
        audio_path: Optional[str] = None,
        transcript_path: Optional[str] = None,
        dlp_redact_template: Optional[str] = None,
        dlp_deidentify_template: Optional[str] = None,
        auth: Optional[base.Auth] = None,
        config: Optional[base.Config] = None,
    ) -> None:
        """Initializes the Ingestion client.

        Args:
            parent: The parent resource name (e.g., 'projects/p/locations/l').
            audio_path: The GCS URI of the audio file.
            transcript_path: The GCS URI of the transcript file.
            dlp_redact_template: The DLP inspect template for redaction.
            dlp_deidentify_template: The DLP de-identify template.
            auth: An optional authentication object.
            config: An optional configuration object.
        """
        self.auth = auth or base.Auth()
        self.config = config or base.Config()
        self.parent = parent
        self.audio_path = audio_path
        self.transcript_path = transcript_path
        self.dlp_redact_template = dlp_redact_template
        self.dlp_deidentify_template = dlp_deidentify_template
        self.client = contact_center_insights_v1.ContactCenterInsightsClient(
            client_options=self.config.set_insights_endpoint()
        )

    def _generate_conversation_id(self) -> str:
        """Generates a random conversation ID."""
        return str(randint(10000000000000000, 999999999999999999))

    def _set_upload_conversation_request(
        self,
        conversation_id: str,
        conversation: types.Conversation,
    ) -> types.UploadConversationRequest:
        """Create the request for Upload Conversation
        
        Args:
            conversation_id: The ID of the conversation.
            conversation: The conversation object.
            
        Returns:
            The upload conversation request.
        """
        if not conversation_id or conversation_id == "None":
            conversation_id = self._generate_conversation_id()

        req = types.UploadConversationRequest(
            parent=self.parent,
            conversation_id=conversation_id,
            conversation=conversation,
        )

        if self.dlp_redact_template:
            req.redaction_config.inspect_template = self.dlp_redact_template

        if self.dlp_deidentify_template:
            req.redaction_config.deidentify_template = self.dlp_deidentify_template
        return req

    def _set_conversation(
        self,
        language_code: str = "en-US",
        medium: Mediums = Mediums.PHONE_CALL,
        audio_uri: Optional[str] = None,
        transcript_uri: Optional[str] = None,
        agent: Optional[List[Dict[str, str]]] = None,
        labels: Optional[MutableMapping[str, str]] = None,
        start_time: Optional[Timestamp] = None,
        customer_satisfaction: Optional[int] = None,
        agent_type: AgentType = AgentType.HUMAN_AGENT
    ) -> types.Conversation:
        """Creates a Conversation object.

        Args:
            language_code: The language of the conversation.
            medium: The medium of the conversation.
            audio_uri: The GCS URI of the audio file.
            transcript_uri: The GCS URI of the transcript file.
            agent: A list of agent information dictionaries.
            labels: A dictionary of labels for the conversation.
            start_time: The start time of the conversation.
            customer_satisfaction: The customer satisfaction rating.
            agent_type: The type of agent.
            
        Returns:
            A Conversation object.
        """
        gcs_source_kwargs = {}
        if audio_uri:
            gcs_source_kwargs["audio_uri"] = audio_uri
            if not start_time:
                start_time = Timestamp()
                start_time.GetCurrentTime()
        elif transcript_uri:
            gcs_source_kwargs["transcript_uri"] = transcript_uri
            if not start_time: # Assume transcript implies a past conversation, but requires start_time
                start_time = Timestamp()
                start_time.GetCurrentTime()
        else:
            raise ValueError("Either audio_uri or transcript_uri must be provided")

        convo = types.Conversation(
            start_time=start_time,
            medium=medium.value,
            language_code=language_code,
            quality_metadata=types.Conversation.QualityMetadata(
                agent_info=[]
            ),
            data_source=types.ConversationDataSource(
                gcs_source=types.GcsSource(**gcs_source_kwargs)
            ),
        )

        if agent:
            for agent_data in agent:
                agent_info = (
                    types.Conversation.QualityMetadata.AgentInfo(
                        agent_type=agent_type.value
                    )
                )
                if "name" in agent_data:
                    agent_info.display_name = agent_data["name"]

                if "id" in agent_data:
                    agent_info.agent_id = agent_data["id"]

                if "team" in agent_data:
                    agent_info.team = agent_data["team"]

                # pylint: disable-next=no-member
                convo.quality_metadata.agent_info.append(agent_info)

        if labels:
            convo.labels = labels

        if customer_satisfaction:
            convo.quality_metadata.customer_satisfaction_rating = customer_satisfaction

        return convo

    def _set_ingest_conversations_request(
        self, metadata: Optional[str] = None, medium: Mediums = Mediums.PHONE_CALL
    ) -> types.IngestConversationsRequest:
        """Creates an IngestConversationsRequest.
 
        Args:
            metadata: The GCS URI of the metadata file.
            medium: The medium of the conversations.

        Returns:
            An IngestConversationsRequest object.
        """
        if not self.transcript_path:
            raise ValueError("transcript_path must be provided for bulk ingestion")

        req = types.IngestConversationsRequest(
            parent=self.parent,
            gcs_source=types.IngestConversationsRequest.GcsSource(
                bucket_uri=self.transcript_path,

            ),
            transcript_object_config=types.IngestConversationsRequest.TranscriptObjectConfig(
                medium=medium.value
            ),
        )

        if metadata:
            req.gcs_source.metadata_bucket_uri = metadata

        if self.dlp_redact_template:
            req.redaction_config.inspect_template = self.dlp_redact_template

        if self.dlp_deidentify_template:
            req.redaction_config.deidentify_template = self.dlp_deidentify_template

        return req

    def single(
        self,
        language_code: str = "en-US",
        medium: Mediums = Mediums.PHONE_CALL,
        conversation_id: Optional[str] = None,
        agent: Optional[List[Dict[str, str]]] = None,
        labels: Optional[MutableMapping[str, str]] = None,
        customer_satisfaction: Optional[int] = None,
    ) -> Operation:
        """Ingests a single conversation.

        Args:
            language_code: The language of the conversation.
            medium: The medium of the conversation.
            conversation_id: The ID for the conversation.
            agent: A list of agent information dictionaries.
            labels: A dictionary of labels for the conversation.
            customer_satisfaction: The customer satisfaction rating.

        Returns:
            A long-running operation for the ingestion process.
        """
        convo = self._set_conversation(
            audio_uri=self.audio_path,
            transcript_uri=self.transcript_path,
            language_code=language_code,
            medium=medium,
            agent=agent,
            labels=labels,
            customer_satisfaction=customer_satisfaction,
        )
        return self.client.upload_conversation(
            request=self._set_upload_conversation_request(
                conversation_id=str(conversation_id), conversation=convo
            )
        )

    def bulk(
        self, metadata_path: Optional[str] = None, medium: Mediums = Mediums.PHONE_CALL
    ) -> Operation:
        """Ingests multiple conversations in bulk.

        Args:
            metadata_path: The GCS URI of the metadata file.
            medium: The medium of the conversations.

        Returns:
            A long-running operation for the ingestion process.
        """
        return self.client.ingest_conversations(
            request=self._set_ingest_conversations_request(
                metadata=metadata_path, medium=medium
            )
        )

class Analysis:
    """Performs analysis on conversations."""

    def __init__(
        self,
        parent: str,
        auth: Optional[base.Auth] = None,
        config: Optional[base.Config] = None,
    ) -> None:
        """Initializes the Analysis client.

        Args:
            parent: The parent resource name (e.g., 'projects/p/locations/l/conversations/c').
            auth: An optional authentication object.
            config: An optional configuration object.
        """
        self.auth = auth or base.Auth()
        self.config = config or base.Config()

        self.parent = parent
        self.client = contact_center_insights_v1.ContactCenterInsightsClient(
            client_options=self.config.set_insights_endpoint()
        )

    def _set_annotators(
        self,
        annotators: List[str],
    ) -> types.AnnotatorSelector:
        """Configures annotators for analysis.
        
        Args:
            annotators: A list of annotators to enable.

        Returns:
            An AnnotatorSelector object with the specified annotators enabled.
        """

        selected_annotators = contact_center_insights_v1.AnnotatorSelector(
            run_interruption_annotator=False,
            run_silence_annotator=False,
            run_phrase_matcher_annotator=False,
            run_sentiment_annotator=False,
            run_entity_annotator=False,
            run_intent_annotator=False,
            run_issue_model_annotator=False,
            run_summarization_annotator=False,
        )

        for annotator in annotators:
            if annotator == Annotators.SUMMARIZATION:
                selected_annotators.run_summarization_annotator = True
            if annotator == Annotators.TOPIC_MODEL:
                selected_annotators.run_issue_model_annotator = True
            if annotator == Annotators.INSIGHTS:
                selected_annotators.run_intent_annotator = True
                selected_annotators.run_entity_annotator = True
                selected_annotators.run_sentiment_annotator = True
                selected_annotators.run_phrase_matcher_annotator = True
                selected_annotators.run_silence_annotator = True
                selected_annotators.run_interruption_annotator = True
            if annotator == Annotators.QAI:
                raise ValueError("QAI annotator is not available")
        return selected_annotators

    def single(
        self,
        annotators: List[str],
    ) -> Operations:
        """Creates an analysis for a single conversation.

        Args:
            annotators: A list of annotators to use for the analysis.

        Returns:
            A long-running operation for the analysis creation.
        """
        return self.client.create_analysis(
            request=types.CreateAnalysisRequest(
                parent=self.parent,
                analysis=types.Analysis(
                    name=self.parent,
                    annotator_selector=self._set_annotators(annotators),
                ),
            )
        )

    def bulk(
        self,
        annotators: list,
        analysis_percentage: float,
        analysis_filter: str,
    ) -> Operation:
        """Exports conversation data from Contact Center AI Insights.

        Args:
            annotators: A list of annotators to use for the analysis.
            analysis_percentage: The percentage of conversations to analyze.
            analysis_filter: A filter to apply to the conversations.

        Returns:
            A long-running operation for the bulk analysis.
        """
        result = self.client.bulk_analyze_conversations(
            request=types.BulkAnalyzeConversationsRequest(
                parent=self.parent,
                filter=analysis_filter,
                analysis_percentage=analysis_percentage,
                annotator_selector=self._set_annotators(annotators),
            )
        )
        return result

class Export:
    """Class for all native export actions from Insights"""

    def __init__(
        self,
        parent: str,
        auth: Optional[base.Auth] = None,
        config: Optional[base.Config] = None,
    ) -> None:
        """Initializes the Export client.

        Args:
            parent: The parent resource name (e.g., 'projects/p/locations/l').
            auth: An optional authentication object.
            config: An optional configuration object.
        """
        self.auth = auth or base.Auth()
        self.config = config or base.C onfig()

        self.parent = parent
        self.client = contact_center_insights_v1.ContactCenterInsightsClient(
            client_options=self.config.set_insights_endpoint()
        )

    def to_bq(
        self,
        project_id: str,
        dataset: str,
        table: str,
        insights_filter: Optional[str] = None,
    ) -> Operation:
        """Exports insights data to BigQuery.

        Args:
            project_id: The BigQuery project ID.
            dataset: The BigQuery dataset ID.
            table: The BigQuery table ID.
            insights_filter: An optional filter for the insights data.

        Returns:
            A long-running operation for the export process.
        """
        return self.client.export_insights_data(
            request=types.ExportInsightsDataRequest(
                parent=self.parent,
                filter=insights_filter,
                big_query_destination=types.ExportInsightsDataRequest.BigQueryDestination(
                    project_id=project_id, dataset=dataset, table=table
                ),
            )
        )

class AuthorizedViews:
    """Manages authorized views for Contact Center AI Insights."""

    def __init__(
        self,
        parent: str,
        auth: Optional[base.Auth] = None,
        config: Optional[base.Config] = None,
    ) -> None:
        """Initializes the AuthorizedViews client.
 
        Args:
            parent: The Google Cloud project ID and location
            auth: An optional authentication object.
            config: An optional configuration object.
        """

        self.auth = auth or base.Auth()
        self.config = config or base.Config()
        self.parent = parent
        parts = self.parent.split('/')
        self.project_id = parts[1]
        self.location = parts[3]
        self._view_sets_endpoint = "authorizedViewSets"

        self.request = base.Request(
            project_id=self.project_id,
            location=self.location,
            auth=self.auth,
            config=self.config,
            base_url=f"https://contactcenterinsights.googleapis.com/v1/{self.parent}/"
        )

    def _make_request(self, *args, **kwargs) -> Optional[Dict]:
        """Makes a request and returns the JSON response."""
        response = self.request.make(*args, **kwargs)
        return response.json() if response else None

    def create_view_set(
        self,
        authorized_view_set_name: str,
    ) -> Optional[Dict]:
        """Creates a new Authorized Views Set
        
        Args:
            authorized_view_set_name: The name of the authorized view set.
            
        Returns:
            The created authorized view set.
        """
        return self._make_request(
            endpoint=self._view_sets_endpoint,
            method=base.Methods.POST,
            payload={
                "displayName": authorized_view_set_name,
            },
        )

    def list_view_set(
        self,
        params: Optional[Dict[str, str]] = None,
    ) -> Optional[Dict]:
        """Lists Authorized Views Sets
        
        Args:
            params: A dictionary of parameters to include in the request.
            
        Returns:
            The list of authorized view sets.
        """
        return self._make_request(
            endpoint=self._view_sets_endpoint,
            method=base.Methods.GET,
            payload=params,
        )

    def get_view_set(
        self,
        view_set_id: str,
        params: Optional[Dict[str, str]] = None,
    ) -> Optional[Dict]:
        """Gets an Authorized Views Set
        
        Args:
            view_set_id: The ID of the authorized view set.
            params: A dictionary of parameters to include in the request.
            
        Returns:
            The authorized view set.
        """
        endpoint = f"{self._view_sets_endpoint}/{view_set_id}"
        return self._make_request(
            endpoint=endpoint,
            method=base.Methods.GET,
            payload=params
        )

    def delete_view_set(
        self,
        authorized_view_set_id: str,
    ) -> Optional[Dict]:
        """Deletes an Authorized Views Set
        
        Args:
            authorized_view_set_id: The ID of the authorized view set.
            
        Returns:
            An empty dictionary if successful.
        """
        endpoint = f"{self._view_sets_endpoint}/{authorized_view_set_id}"
        return self._make_request(
            endpoint=endpoint,
            method=base.Methods.DELETE,
            payload={}
        )

    def create_view(
        self,
        authorized_view_set_id: str,
        display_name: str,
        conversation_filter: Optional[str] = None,
    ) -> Optional[Dict]:
        """Creates a new authorized view.

        Args:
            authorized_view_set_id: The ID of the authorized view set.
            display_name: The display name of the view.
            conversation_filter: The filter to apply to conversations.
            
        Returns:
            The created authorized view.
        """
        endpoint = f"{self._view_sets_endpoint}/{authorized_view_set_id}/authorizedViews"
        payload = {
                "displayName": display_name,
        }
        if conversation_filter:
            payload["conversationFilter"] = conversation_filter

        return self._make_request(
            endpoint=endpoint,
            method=base.Methods.POST,
            payload=payload,
        )

    def list_view(
        self,
        authorized_view_set_id: str,
        params: Optional[Dict[str, str]] = None,
    ) -> Optional[Dict]:
        """Lists authorized views within a set.

        Args:
            authorized_view_set_id: The ID of the authorized view set.
            params: A dictionary of parameters to include in the request.

        Returns:
            The list of authorized views.
        """
        endpoint = f"{self._view_sets_endpoint}/{authorized_view_set_id}/authorizedViews"
        return self._make_request(
            endpoint=endpoint,
            method=base.Methods.GET,
            payload=params,
        )

    def get_view(
        self,
        authorized_view_set_id: str,
        authorized_view_id: str,
        params: Optional[Dict[str, str]] = None,
    ) -> Optional[Dict]:
        """Gets an authorized view.

        Args:
            authorized_view_set_id: The ID of the authorized view set.
            authorized_view_id: The ID of the authorized view.
            params: A dictionary of parameters to include in the request.

        Returns:
            The authorized view.
        """
        endpoint = (
            f"{self._view_sets_endpoint}/{authorized_view_set_id}"
            f"/authorizedViews/{authorized_view_id}"
        )
        return self._make_request(
            endpoint=endpoint,
            method=base.Methods.GET,
            payload=params,
        )

    def delete_view(
        self,
        authorized_view_set_id: str,
        authorized_view_id: str,
    ) -> Optional[Dict]:
        """Deletes an authorized view.
        
        Args:
            authorized_view_set_id: The ID of the authorized view set.
            authorized_view_id: The ID of the authorized view.
            
        Returns:
            An empty dictionary if successful.
        """
        endpoint = (
            f"{self._view_sets_endpoint}/{authorized_view_set_id}"
            f"/authorizedViews/{authorized_view_id}"
        )
        return self._make_request(
            endpoint=endpoint,
            method=base.Methods.DELETE,
            payload={}
        )

class LongRunningOperation:
    """A class to handle long runing operations"""

    def __init__(
        self,
        project_id: str,
        location: str,
        operaton_id: Optional[str] = None,
        auth: Optional[base.Auth] = None,
        config: Optional[base.Config] = None,
    ):
        """Initializes the LongRunningOperation client.
        
        Args:
            project_id: The Google Cloud project ID.
            location: The location of the operation.
            operaton_id: The ID of the operation.
            auth: An optional authentication object.
            config: An optional configuration object.
        """
        self.project_id = project_id
        self.location = location
        self.operation_id = operaton_id
        self.auth = auth or base.Auth()
        self.config = config or base.Config()

        self.operation_name = (
            f"projects/{project_id}/locations/{self.location}/operations/{self.operation_id}"
            if self.operation_id
            else None
        )
        self.client = contact_center_insights_v1.ContactCenterInsightsClient(
            client_options=self.config.set_insights_endpoint()
        )

    def cancel(
        self,
    )->None:
        """Cancels an operation
        
        Raises:
            RuntimeError: If no operation name is provided.
        """
        if not self.operation_name:
            raise RuntimeError("No operation name provided")

        cancel_request = CancelOperationRequest(name=self.operation_name)
        # In a real scenario, this would send the cancel request to the API.
        # For this mock, we'll just print a message.
        print(f"[Mock] Attempting to cancel operation: {self.operation_name}")
        # self.client.cancel_operation(request=cancel_request) # Uncomment for live API interaction

## Core Enums and Their Usage
The `insights.py` module defines several enums to standardize various parameters used in the API interactions, improving readability and reducing errors. Let's look at them.

In [ ]:
print("--- Masks Enum (for update_mask in API requests) ---")
for mask in Masks:
    print(f"  {mask.name}: {mask.value}")

print("\n--- Annotators Enum (types of analysis to perform) ---")
for annotator in Annotators:
    print(f"  {annotator.name}: {annotator.value}")

print("\n--- AgentType Enum (role of conversation participant) ---")
for agent_type in AgentType:
    print(f"  {agent_type.name}: {agent_type.value}")

print("\n--- Mediums Enum (type of conversation) ---")
for medium in Mediums:
    print(f"  {medium.name}: {medium.value}")

## 1. Managing Insights Settings (`Settings` Class)
The `Settings` class allows you to configure global parameters for your CCAI Insights project. These settings include auto-analysis percentages, data retention policies (TTL), Pub/Sub notifications, default language, speech recognizers, and DLP configurations.

In [ ]:
# Initialize the Settings client
# The 'parent' for settings is typically projects/{PROJECT_ID}/locations/{LOCATION}/settings
insights_settings = Settings(project_id=PROJECT_ID, parent=PARENT)
print("Settings client initialized.")

### Get Current Settings
Retrieve the current global settings for your project. This is a good way to see the current configuration before making changes.

In [ ]:
current_settings = insights_settings.get()
print("Current CCAI Insights Settings:")
print(current_settings)

### Update Global Auto-Analysis Settings
Configure how often runtime and uploaded conversations are automatically analyzed, and which annotators to use for this automatic analysis. Percentages are floats between 0.0 and 1.0.

In [ ]:
updated_auto_analysis_settings = insights_settings.update_global_auto_analysis(
    runtime_percentage=0.8, # 80% of runtime conversations will be automatically analyzed
    upload_percentage=0.5,  # 50% of uploaded conversations will be automatically analyzed
    analysis_annotators=[Annotators.INSIGHTS, Annotators.SUMMARIZATION] # Enable core insights and summarization annotators
)
print("Global auto-analysis settings updated:")
print(updated_auto_analysis_settings.analysis_config)

### Update Conversation TTL (Time-To-Live)
Set the data retention period for conversations. Conversations and their analyses older than the specified TTL will be automatically deleted to comply with data retention policies.

In [ ]:
updated_ttl_settings = insights_settings.update_ttl(ttl_in_days=30) # Retain data for 30 days
print("Conversation TTL updated:")
print(f"  Conversation TTL (seconds): {updated_ttl_settings.conversation_ttl.seconds}")

### Update Pub/Sub Notification Settings
Configure Pub/Sub topics to receive notifications for various events (e.g., when a new conversation is ingested or an analysis is completed). This is crucial for event-driven architectures.

In [ ]:
PUB_SUB_TOPIC_CONVERSATION = "projects/YOUR_PROJECT_ID/topics/YOUR_CONVERSATION_TOPIC" # @param {type:"string"}
PUB_SUB_TOPIC_ANALYSIS = "projects/YOUR_PROJECT_ID/topics/YOUR_ANALYSIS_TOPIC" # @param {type:"string"}

pubsub_map = {
    "conversation": PUB_SUB_TOPIC_CONVERSATION,
    "analysis": PUB_SUB_TOPIC_ANALYSIS
}

updated_pubsub_settings = insights_settings.update_pubsub(pub_sub_map)
print("Pub/Sub notification settings updated:")
print(updated_pubsub_settings.pubsub_notification_settings)

if "YOUR_CONVERSATION_TOPIC" in PUB_SUB_TOPIC_CONVERSATION:
    print("\n*** IMPORTANT: Please update YOUR_CONVERSATION_TOPIC and YOUR_ANALYSIS_TOPIC with your actual Pub/Sub topics. ***")

### Update Global Language Code
Set the default language code for new conversations that don't explicitly specify one. This helps ensure proper transcription and analysis.

In [ ]:
updated_language_settings = insights_settings.update_global_language(language_code="es-US") # Example: Set default language to US Spanish
print("Global language code updated:")
print(f"  Language Code: {updated_language_settings.language_code}")

### Update Global Speech Recognizer
Specify a custom speech recognizer to be used for transcription. This is useful if you have specialized vocabulary or acoustic models.

In [ ]:
SPEECH_RECOGNIZER_PATH = "projects/YOUR_PROJECT_ID/locations/global/speechRecognizers/YOUR_RECOGNIZER_ID" # @param {type:"string"}

updated_speech_settings = insights_settings.update_global_speech(speech_recognizer_path=SPEECH_RECOGNIZER_PATH)
print("Global speech recognizer updated:")
print(f"  Speech Recognizer: {updated_speech_settings.speech_config.speech_recognizer}")

if "YOUR_RECOGNIZER_ID" in SPEECH_RECOGNIZER_PATH:
    print("\n*** IMPORTANT: Please update YOUR_RECOGNIZER_ID with the resource path to your custom speech recognizer. ***")

### Update Global DLP (Data Loss Prevention) Settings
Configure DLP templates for redacting or de-identifying sensitive information from conversations before they are stored or analyzed. This is critical for privacy and compliance.

In [ ]:
DLP_INSPECT_TEMPLATE = "projects/YOUR_PROJECT_ID/locations/global/inspectTemplates/YOUR_INSPECT_TEMPLATE_ID" # @param {type:"string"}
DLP_DEIDENTIFY_TEMPLATE = "projects/YOUR_PROJECT_ID/locations/global/deidentifyTemplates/YOUR_DEIDENTIFY_TEMPLATE_ID" # @param {type:"string"}

updated_dlp_settings = insights_settings.update_global_dlp(
    inspect_template=DLP_INSPECT_TEMPLATE,
    deidentify_template=DLP_DEIDENTIFY_TEMPLATE
)
print("Global DLP settings updated:")
print(f"  Inspect Template: {updated_dlp_settings.redaction_config.inspect_template}")
print(f"  De-identify Template: {updated_dlp_settings.redaction_config.deidentify_template}")

if "YOUR_INSPECT_TEMPLATE_ID" in DLP_INSPECT_TEMPLATE:
    print("\n*** IMPORTANT: Please update YOUR_INSPECT_TEMPLATE_ID and YOUR_DEIDENTIFY_TEMPLATE_ID with your actual DLP template resource paths. ***")

## 2. Ingesting Conversation Data (`Ingestion` Class)
The `Ingestion` class facilitates uploading conversation data into CCAI Insights, either as single audio/transcript files or in bulk via Google Cloud Storage (GCS) buckets. Ingested conversations are then available for analysis.

### Prepare GCS URIs
For ingestion, you'll need GCS URIs pointing to your conversation data. These are placeholders; replace them with actual paths to your files. For a runnable example, a public sample audio URI is provided.

In [ ]:
GCS_AUDIO_URI = "gs://cloud-samples-data/speech/conversation_sample.wav" # Public sample audio URI
GCS_TRANSCRIPT_URI = "gs://YOUR_BUCKET/conversations/transcripts.csv" # @param {type:"string"} Placeholder for your bulk transcript file
GCS_METADATA_URI = "gs://YOUR_BUCKET/conversations/metadata.jsonl" # @param {type:"string"} Placeholder for your bulk ingestion metadata file

print(f"GCS Audio URI (for single ingestion): {GCS_AUDIO_URI}")
print(f"GCS Transcript URI (for bulk ingestion): {GCS_TRANSCRIPT_URI}")
print(f"GCS Metadata URI (for bulk ingestion): {GCS_METADATA_URI}")

if "YOUR_BUCKET" in GCS_TRANSCRIPT_URI:
    print("\n*** IMPORTANT: Please update YOUR_BUCKET with your actual GCS bucket name for bulk ingestion examples. ***")

In [ ]:
# Initialize the Ingestion client
# We'll set audio_path for single ingestion example, and transcript_path for bulk.
insights_ingestion = Ingestion(
    parent=PARENT,
    audio_path=GCS_AUDIO_URI,
    transcript_path=GCS_TRANSCRIPT_URI
)
print("Ingestion client initialized.")

### Ingest a Single Conversation
Upload a single audio or transcript file for analysis. This operation returns a long-running operation (`Operation` object) that you can monitor for completion.

In [ ]:
try:
    single_conversation_op = insights_ingestion.single(
        conversation_id="my-unique-conversation-123",
        language_code="en-US",
        medium=Mediums.PHONE_CALL,
        agent=[{"name": "Support Agent", "id": "agent-456"}],
        labels={"department": "sales", "priority": "high"},
        customer_satisfaction=5 # Rating from 1 to 5
    )
    print("Single conversation ingestion initiated. Operation Name:")
    print(single_conversation_op.operation.name)
    # For a real scenario, you'd poll single_conversation_op for completion.

except ValueError as e:
    print(f"Error during single conversation ingestion: {e}")
    print("Please ensure `GCS_AUDIO_URI` is set to a valid path for single ingestion with audio, or `GCS_TRANSCRIPT_URI` for transcript.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

### Ingest Multiple Conversations in Bulk
Upload multiple conversations using a GCS bucket URI pointing to a set of transcript files and an optional metadata file. This also returns a long-running operation (`Operation` object).

In [ ]:
try:
    # Note: For bulk ingestion, `insights_ingestion` was initialized with `transcript_path=GCS_TRANSCRIPT_URI`.
    # The `GCS_TRANSCRIPT_URI` must point to a GCS bucket containing transcript files.
    bulk_ingestion_op = insights_ingestion.bulk(
        metadata_path=GCS_METADATA_URI, # Optional: GCS URI to a metadata file (e.g., JSONL format)
        medium=Mediums.CHAT
    )
    print("Bulk conversation ingestion initiated. Operation Name:")
    print(bulk_ingestion_op.operation.name)
    # For a real scenario, you'd poll bulk_ingestion_op for completion.
except ValueError as e:
    print(f"Error during bulk ingestion: {e}")
    print("Please ensure `GCS_TRANSCRIPT_URI` is set to a valid bucket URI for bulk ingestion, e.g., 'gs://your-bucket/transcripts/'.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

## 3. Performing Conversation Analysis (`Analysis` Class)
The `Analysis` class allows you to trigger analysis on conversations, leveraging various annotators to extract insights such as sentiment, entities, intents, and topics. You can analyze single conversations or perform bulk analysis across many.

### Prepare a Conversation Resource Name
To perform analysis on a single conversation, you need the full resource name of an already ingested conversation. You would typically obtain this from the output of an ingestion operation or by listing existing conversations in CCAI Insights.

In [ ]:
# Placeholder for an existing conversation resource name
# Replace 'your-conversation-id-here' with an actual ID from your ingested conversations.
CONVERSATION_NAME_FOR_ANALYSIS = f"projects/{PROJECT_ID}/locations/{LOCATION}/conversations/your-conversation-id-here" # @param {type:"string"}

# Initialize the Analysis client for a single conversation
insights_analysis_single = Analysis(parent=CONVERSATION_NAME_FOR_ANALYSIS)
print("Analysis client for single conversation initialized.")

if "your-conversation-id-here" in CONVERSATION_NAME_FOR_ANALYSIS:
    print("\n*** IMPORTANT: Please update 'your-conversation-id-here' with an actual conversation ID. ***")

### Create a Single Analysis
Trigger analysis for a specific conversation with a chosen set of annotators. This returns a long-running operation that you can monitor for the analysis results.

In [ ]:
try:
    # Note: Analysis.single's return type hint is Operations, but typically returns Operation.
    # Accessing .name property of the underlying operation object.
    single_analysis_op = insights_analysis_single.single(
        annotators=[Annotators.INSIGHTS, Annotators.TOPIC_MODEL] # Enable general insights and topic modeling
    )
    print("Single analysis initiated. Operation Name:")
    print(single_analysis_op.name)
    # For a real scenario, you'd poll single_analysis_op for completion.

except ValueError as e:
    print(f"Error: {e}")
    print("Note: The QAI annotator is explicitly not available for single analysis in this module.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

### Perform Bulk Analysis
Analyze multiple conversations across your project based on a filter and a percentage of conversations to include. This is particularly useful for analyzing large datasets efficiently.

In [ ]:
# For bulk analysis, the parent should be the location, not a specific conversation.
insights_analysis_bulk = Analysis(parent=PARENT)

try:
    bulk_analysis_op = insights_analysis_bulk.bulk(
        annotators=[Annotators.INSIGHTS, Annotators.SUMMARIZATION],
        analysis_percentage=0.75, # Analyze 75% of filtered conversations
        analysis_filter="create_time > \"2023-01-01T00:00:00Z\" AND medium = \"PHONE_CALL\"" # Example: filter for calls after a specific date
    )
    print("Bulk analysis initiated. Operation Name:")
    print(bulk_analysis_op.operation.name)
    # For a real scenario, you'd poll bulk_analysis_op for completion.
except Exception as e:
    print(f"An error occurred during bulk analysis: {e}")

## 4. Exporting Insights Data (`Export` Class)
The `Export` class provides functionality to export processed insights data to other Google Cloud services, such as BigQuery, for deep analytics, custom dashboards, or integration with other data pipelines.

In [ ]:
# Initialize the Export client. The parent is the project/location.
insights_export = Export(parent=PARENT)
print("Export client initialized.")

### Export to BigQuery
Export conversation insights data to a specified BigQuery dataset and table. You can also apply a filter to export a subset of data, making it flexible for various reporting needs.

In [ ]:
BIGQUERY_DATASET = "YOUR_BIGQUERY_DATASET_ID" # @param {type:"string"}
BIGQUERY_TABLE = "your_insights_table" # @param {type:"string"}

export_bq_op = insights_export.to_bq(
    project_id=PROJECT_ID,
    dataset=BIGQUERY_DATASET,
    table=BIGQUERY_TABLE,
    insights_filter="conversation_start_time > \"2024-01-01T00:00:00Z\"" # Optional filter: export data for conversations starting after Jan 1, 2024
)
print("Export to BigQuery initiated. Operation Name:")
print(export_bq_op.operation.name)
# For a real scenario, you'd poll export_bq_op for completion.

if "YOUR_BIGQUERY_DATASET_ID" in BIGQUERY_DATASET:
    print("\n*** IMPORTANT: Please update YOUR_BIGQUERY_DATASET_ID and 'your_insights_table' with your actual BigQuery dataset and table names. ***")

## 5. Managing Authorized Views (`AuthorizedViews` Class)
The `AuthorizedViews` class allows you to create and manage view sets and views, which provide granular access control to subsets of your conversation data for different users or applications. This is crucial for security and compliance, ensuring users only see relevant data.

In [ ]:
# Initialize the AuthorizedViews client. The parent is the project/location.
insights_views = AuthorizedViews(parent=PARENT)
print("AuthorizedViews client initialized.")

### Create an Authorized View Set
A view set acts as a logical container for related authorized views. You typically create one for each team or purpose that requires specific data access.

In [ ]:
VIEW_SET_DISPLAY_NAME = "MyMarketingTeamViewSet" # @param {type:"string"}
created_view_set = insights_views.create_view_set(authorized_view_set_name=VIEW_SET_DISPLAY_NAME)
print("Created Authorized View Set:")
print(created_view_set)

# Extract the actual view set ID from the mock response for subsequent calls
mock_view_set_id = created_view_set['name'].split('/')[-1] if created_view_set and 'name' in created_view_set else 'mock-view-set-1'
print(f"Extracted View Set ID: {mock_view_set_id}")

### List Authorized View Sets
Retrieve a list of all authorized view sets configured in your project. This helps in auditing and management.

In [ ]:
list_of_view_sets = insights_views.list_view_set()
print("List of Authorized View Sets:")
print(list_of_view_sets)

### Get an Authorized View Set
Retrieve detailed information for a specific authorized view set by its ID.

In [ ]:
get_view_set_details = insights_views.get_view_set(view_set_id=mock_view_set_id)
print(f"Details for View Set '{mock_view_set_id}':")
print(get_view_set_details)

### Create an Authorized View
Create a specific view within a view set. Each view defines a `conversation_filter` that determines which conversations are visible through this view. This is where you define the specific subset of data a user or application can access.

In [ ]:
VIEW_DISPLAY_NAME = "HighSatisfactionCallsView" # @param {type:"string"}
CONVERSATION_FILTER_FOR_VIEW = "quality_metadata.customer_satisfaction_rating >= 4" # @param {type:"string"} Example: Filter for calls with high satisfaction

created_view = insights_views.create_view(
    authorized_view_set_id=mock_view_set_id,
    display_name=VIEW_DISPLAY_NAME,
    conversation_filter=CONVERSATION_FILTER_FOR_VIEW
)
print("Created Authorized View:")
print(created_view)

# Extract the actual view ID from the mock response for subsequent calls
mock_view_id = created_view['name'].split('/')[-1] if created_view and 'name' in created_view else 'mock-view-1'
print(f"Extracted View ID: {mock_view_id}")

### List Authorized Views
List all authorized views that belong to a specific view set.

In [ ]:
list_of_views = insights_views.list_view(authorized_view_set_id=mock_view_set_id)
print(f"List of Authorized Views in '{mock_view_set_id}':")
print(list_of_views)

### Get an Authorized View
Retrieve detailed information for a specific authorized view by its ID within a given view set.

In [ ]:
get_view_details = insights_views.get_view(authorized_view_set_id=mock_view_set_id, authorized_view_id=mock_view_id)
print(f"Details for View '{mock_view_id}' in '{mock_view_set_id}':")
print(get_view_details)

### Delete an Authorized View
Remove a specific authorized view. This revokes access to the data subset defined by that view.

In [ ]:
delete_view_response = insights_views.delete_view(authorized_view_set_id=mock_view_set_id, authorized_view_id=mock_view_id)
print(f"Deletion response for view '{mock_view_id}':")
print(delete_view_response)

### Delete an Authorized View Set
Remove an entire authorized view set. You typically delete views before their parent view set, but our mock allows direct deletion for simplicity.

In [ ]:
delete_view_set_response = insights_views.delete_view_set(authorized_view_set_id=mock_view_set_id)
print(f"Deletion response for view set '{mock_view_set_id}':")
print(delete_view_set_response)

## 6. Handling Long-Running Operations (`LongRunningOperation` Class)
Many CCAI Insights API calls (e.g., ingestion, analysis, export) are asynchronous and return a `LongRunningOperation` object. The `LongRunningOperation` class in this module provides a way to manage these operations, primarily for cancellation.

### Initializing and Mocking an Operation
To demonstrate, we'll initialize the `LongRunningOperation` class with a mock operation ID. In a real application, this ID would be extracted from the `.operation.name` attribute of an `Operation` object returned by an API call (like those shown in the `Ingestion`, `Analysis`, and `Export` sections).

In [ ]:
# Example: Let's assume this is an operation ID obtained from an ingestion or analysis call
MOCK_OPERATION_ID = "1234567890-mock-operation" # @param {type:"string"}

# Initialize the LongRunningOperation client with the operation ID
lro_handler = LongRunningOperation(
    project_id=PROJECT_ID,
    location=LOCATION,
    operaton_id=MOCK_OPERATION_ID
)
print("LongRunningOperation client initialized.")
print(f"Operation Name: {lro_handler.operation_name}")

### Canceling an Operation
You can cancel a running operation if it's no longer needed or if it's stuck. Note that cancellation is best effort and may not always succeed immediately, depending on the operation's current state.

In [ ]:
try:
    lro_handler.cancel()
    print(f"Operation {MOCK_OPERATION_ID} cancellation requested (mocked).")
except RuntimeError as e:
    print(f"Error canceling operation: {e}")
except Exception as e:
    print(f"An unexpected error occurred during cancellation: {e}")

## Conclusion
This tutorial has walked you through the primary functionalities of the `insights.py` module, covering how to manage settings, ingest and analyze conversation data, export insights, and handle authorized views and long-running operations within Google Cloud Contact Center AI Insights.

By leveraging this module, you can streamline your interactions with the CCAI Insights API, enabling you to build powerful applications for customer experience analysis, agent performance monitoring, and business intelligence. Remember to replace all placeholder values with your actual Google Cloud resource IDs and paths when deploying this code in a production environment.